In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import regridding_utils as ruti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( nuti )
importlib.reload( ruti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")


In [ ]:
#casedic={'basedir':,'case':,'Hkey':}

cases=[]
basedir=f'/glade/derecho/scratch/juliob/archive'
case, Hkey = 'CTL_fmt_c64116'  , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'CTL_fmt_c64144'  , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'CTLre_fmt_c6477' , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/hannay/archive'
case, Hkey = 'f.e30_cam6_4_097.FHISTC_MTso.ne30.001' , 'yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'f.e30_alpha08b.FHISTC_MTso.ne30_t232_wgx3.315' , 'yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'b.e30_alpha08a.B1850C_MTso.ne30_t232_wgx3.280','yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/tilmes/archive'
case, Hkey = 'b.e30_alpha07c_cesm.B1850C_MTt4s.ne30_t232_wgx3.251.001' , 'yx_proc'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/shawnh/archive'
case, Hkey = 'b.e30_beta06_megan.B1850C_MTt4s.ne30.004' , 'yx'
#cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )



In [ ]:
%%time
# Construct structure to look at Q and CH4 in multiple cases
tic_total_0 = time.perf_counter()

mos = [99,] # np.arange( start=1,stop=2)   #13)
Al=[]
for c in cases:
    basedir,case,Hkey = c['basedir'],c['case'],c['Hkey']
    A_={'basedir':basedir,'case':case  , 'Hkey':Hkey } 
    for m in mos:
        if Hkey != 'yx_proc':
            if m != 99:
                filep = f"{basedir}/{case}/atm/hist/{case}.cam.h0a.*-{m:02d}.nc"
            else:
                filep = f"{basedir}/{case}/atm/hist/{case}.cam.h0a.*-*.nc"
            print( f"Begin Read {filep} ")
            X=xr.open_mfdataset( filep ,data_vars='different', coords='different', compat='no_conflicts' )
            print( f"Finsihed Read {filep} ")
            tic_total_1 = time.perf_counter()
            pTime = f"Total reading took  {tic_total_1 - tic_total_0:0.4f} seconds"
            print( pTime )
            X2=X[['lat','lon','PS','Q','CH4']]
            if m == 99 and Hkey=='c':
                X2 = X2.groupby("time.year").mean(dim="time").compute()
                X2['lat'] = X2['lat'].isel(year=0)
                X2['lon'] = X2['lon'].isel(year=0)
                X2 = X2.rename({"year": "time"})
                print( f"Finsihed Group by year ")
                print( f"Size of Q: {np.shape(X2.Q)}" )
                tic_total_2 = time.perf_counter()
                pTime = f"Group by year took  {tic_total_2 - tic_total_1:0.4f} seconds"
                print( pTime )
                #break
            elif m == 99 and Hkey=='yx':
                X2 = X2.groupby("time.year").mean(dim="time").compute()
                X2 = X2.rename({"year": "time"})
                print( f"Finsihed Group by year ")
                print( f"Size of Q: {np.shape(X2.Q)}" )
                tic_total_2 = time.perf_counter()
                pTime = f"Group by year took  {tic_total_2 - tic_total_1:0.4f} seconds"
                print( pTime )
                #break
            else:
                tic_total_2 = time.perf_counter()

        else:
            #/glade/derecho/scratch/tilmes/archive/b.e30_alpha07c_cesm.B1850C_MTt4s.ne30_t232_wgx3.251.001/atm/proc/tseries/month_1
            filep = f"{basedir}/{case}/atm/proc/tseries/month_1/{case}.cam.h0a.Q.*.nc"
            print( filep )
            X=xr.open_mfdataset( filep ,data_vars='different', coords='different', compat='no_conflicts' )
            if m != 99:
                X2 = X.sel(time=X.time.dt.month == m)
            else:
                X2 = X.groupby("time.year").mean(dim="time").compute()
            filep = f"{basedir}/{case}/atm/proc/tseries/month_1/{case}.cam.h0a.CH4.*.nc"
            print( filep )
            X=xr.open_mfdataset( filep ,data_vars='different', coords='different', compat='no_conflicts' )
            if m != 99:
                X2ch4 = X.sel(time=X.time.dt.month == m)
            else:
                X2ch4 = X.groupby("time.year").mean(dim="time").compute()

            X2['CH4']=X2ch4['CH4']
        
        if Hkey == 'c':
            X2r=ruti.regrid_X( X2, Src='ne30pg3', Dst='fv1x1' )
            tic_total_3 = time.perf_counter()
            pTime = f"Total regridding took  {tic_total_3 - tic_total_2:0.4f} seconds"
            print( pTime )
        else:
            X2r = X2
    A_['X']=X2r
    A=AttrDict(A_)
    Al.append(A)


In [ ]:
X2

In [ ]:
X2

In [ ]:
X0_for_jan = Al[0].X

In [ ]:
for A in Al:
    q = A.X.Q.values
    lev=A.X.lev.values
    lat=A.X.lat.values
    lon=A.X.lon.values
    zlev = -7_000. * np.log( lev/ 1_000. )
    q_zv = np.mean( q, axis=3)
    ch4 = A.X.CH4.values
    ch4_zv = np.mean( ch4, axis=3)
    A['ch4_zv']=ch4_zv
    A['q_zv']=q_zv
    A['zlev']=zlev
    A['lon']=lon
    A['lat']=lat
    print( f"{A.case}")
    

In [ ]:
z0s=[15_000.,17_000.,21_000.,25_000.,35_000.,50_000.]
lat0=0.
nplots=len(z0s)
fig,axs = plt.subplots( 2, 3 , figsize=( 8.*nplots/2 , 14. ) )
axs=axs.flatten()
ip=0
for z0 in z0s:
    ax=axs[ip]

    icase=0
    for A in Al:
        q_zv,zlev,lat = A.q_zv,A.zlev,A.lat
        z,y = np.argmin( np.abs( zlev-z0) ),np.argmin( np.abs( lat-lat0) )
        if icase <=4:
            linestyle='-'
        else:
            linestyle = '--'
        ax.plot( q_zv[:,z,y] , label= A.case, linestyle = linestyle )
        icase=icase+1

    ax.set_xlabel('Year since start',fontsize=18)
    ax.set_ylabel("Water Vapor (kg kg$^{-1}$)",fontsize=18)
    ax.set_title(f"Z={z0:05.0f} m" ,fontsize=20 )
    ax.legend()
    if ip>=0:
        ax.set_ylim( 1.e-6*np.asarray( [0.5,4.0] ) )
    ip=ip+1

#plt.legend()



In [ ]:
M_CH4 = 16.04
M_H2O = 18.015

fac = 2.0 * M_H2O / M_CH4   # ~2.245

## QH = H2O + fac * CH4

z0s=[15_000.,17_000.,21_000.,25_000.,35_000.,50_000.]
lat0=0.
nplots=len(z0s)
fig,axs = plt.subplots( 2, 3 , figsize=( 8.*nplots/2 , 14. ) )
axs=axs.flatten()
ip=0
for z0 in z0s:
    ax=axs[ip]

    icase=0
    for A in Al:
        q_zv,ch4_zv,zlev,lat = A.q_zv, A.ch4_zv,  A.zlev,A.lat
        q_H_zv = q_zv + fac*ch4_zv
        z,y = np.argmin( np.abs( zlev-z0) ),np.argmin( np.abs( lat-lat0) )
        if icase <=4:
            linestyle='-'
        else:
            linestyle = '--'
        ax.plot( q_H_zv[:,z,y] , label= A.case, linestyle = linestyle )
        icase=icase+1

    ax.set_xlabel('Year since start',fontsize=18)
    ax.set_ylabel("Methane/Water Vapor Eq (kg kg$^{-1}$)",fontsize=18)
    ax.set_title(f"Z={z0:05.0f} m" ,fontsize=20 )
    ax.legend()
    ip=ip+1

#plt.legend()



In [ ]:
z0s=[15_000.,17_000.,21_000.,25_000.,35_000.,50_000.]
lat0=0.
nplots=len(z0s)
fig,axs = plt.subplots( 2, 3 , figsize=( 8.*nplots/2 , 14. ) )
axs=axs.flatten()
ip=0
for z0 in z0s:
    ax=axs[ip]

    icase=0
    for A in Al:
        q_zv,ch4_zv,zlev,lat = A.q_zv, A.ch4_zv,  A.zlev,A.lat
        z,y = np.argmin( np.abs( zlev-z0) ),np.argmin( np.abs( lat-lat0) )
        if icase <=4:
            linestyle='-'
        else:
            linestyle = '--'
        ax.plot( ch4_zv[:,z,y] , label= A.case, linestyle = linestyle )
        icase=icase+1

    ax.set_xlabel('Year since start',fontsize=18)
    ax.set_ylabel("Methane (kg kg$^{-1}$)",fontsize=18)
    ax.set_title(f"Z={z0:05.0f} m" ,fontsize=20 )
    ax.legend()
    ip=ip+1

#plt.legend()



In [ ]:
print(fac)